In [1]:
import torch
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor
from torch.utils.data import Dataset
import os
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import random
from torch.utils.data import random_split
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import lightning as pl
import torchmetrics
import comet_ml
import os
from lightning.pytorch import Trainer, seed_everything
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from comet_ml import Experiment
import pandas as pd
import numpy as np
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from mediapipe_handcrop import MediapipeHandCrop
from tqdm import tqdm

2025-02-03 16:09:43.766350: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738595383.784677   25280 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738595383.790278   25280 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-03 16:09:43.809441: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## CUDA SETUP

In [2]:
def setup_device():
    if torch.cuda.is_available():
        device = torch.device('cuda')
        torch.set_default_dtype(torch.float32)
    else:
        device = torch.device('cpu')
        torch.set_default_dtype(torch.float32)
    return device

device = setup_device()
print(f"Using {device} device")

# Set random seed for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

Using cuda device


## CREATE TRAIN AND TEST CSV FILES

In [3]:
def average_brightness(image):
    # Convert to CSV
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # # Extracting brightness channel (V)
    brightness_channel = hsv_image[:, :, 2]

    average_brightness = np.mean(brightness_channel)
    if average_brightness > 50: 
        return True
    
    else: return False

In [4]:
train_pth = "./newest_dataset/train"
test_pth = "./newest_dataset/test"

import pandas as pd
import os
import json
from tqdm import tqdm
import cv2
import numpy as np
from PIL import Image

mediapipe_validator = MediapipeHandCrop(include_characteristic_vectors=True)

def validate_dataset_with_mediapipe(dataset_path, csv_file_path, new_dir):
    data = []
    classes_to_ignore = ["del", "nothing", "space"]
    index = 0

    for class_name in os.listdir(dataset_path):
        if class_name not in classes_to_ignore:
            # Create a directory for new validated class images
            new_class_path = os.path.join(new_dir, class_name)
            os.makedirs(new_class_path, exist_ok=True)

            class_index = ord(class_name) - 65
            count = 0

            for image_name in tqdm(os.listdir(os.path.join(dataset_path, class_name))):
                image_path = os.path.join(dataset_path, class_name, image_name)

                # Validate image with Mediapipe and brightness
                try:
                    result = mediapipe_validator(image_path)
                    if result is None:
                        continue
                    cropped_image, characteristic_vectors = result

                    # Check brightness and size
                    if average_brightness(np.array(cropped_image)) and cropped_image.size > (60, 60):
                        # Save the cropped image
                        new_image_path = os.path.join(new_class_path, image_name)
                        cropped_image = np.array(cropped_image)
                        cv2.imwrite(new_image_path, cropped_image)

                        # Append to the dataset
                        data.append({
                            "image_path": new_image_path,
                            "characteristic_vectors": json.dumps(characteristic_vectors),  # Serialize to JSON
                            "class_name": class_name,
                            "class_index": class_index,
                            "index": index
                        })
                        index += 1
                        count += 1

                except Exception as e:
                    print(f"Error processing {image_path}: {e}")
                    continue
                # break

            print(f"Saved {count} images from class {class_name}")

    # Save dataset to a CSV file
    df = pd.DataFrame(data)
    df.to_csv(csv_file_path, index=False)


I0000 00:00:1738595385.369951   25280 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1738595385.412724   25414 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 550.120), renderer: NVIDIA GeForce RTX 3080/PCIe/SSE2


In [5]:
# # Create csv and dir
# validate_dataset_with_mediapipe(train_pth, 
#                                 csv_file_path = r"newest_dataset/validated_train_csv.csv",
#                                 new_dir = r"newest_dataset/validated_train")

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [6]:
# validate_dataset_with_mediapipe(test_pth, 
#                                 csv_file_path = r"newest_dataset/validated_test_csv.csv",
#                                 new_dir = r"newest_dataset/validated_test")

## DATASET

In [7]:
import ast

transform = transforms.Compose([
    transforms.Resize((64,64)),
    # transforms.CenterCrop((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Values calculated from the ImageNet dataset
])

class Dataset(Dataset):
    def __init__(self, root_dir, csv_file, transform=None):
        self.root_dir = root_dir
        self.csv_file = csv_file
        self.transform = transform
        self.df = pd.read_csv(self.csv_file)
        self.classes = sorted(np.unique(self.df["class_index"]))
        self.class_names = sorted(np.unique(self.df["class_name"]))
        
    def __len__(self):
        dataset_len = len(self.df)
        return dataset_len
    
    def __getitem__(self, index):
        if index >= len(self):
            raise IndexError("Index out of bounds")
        
        label = self.df.loc[index, "class_index"]
        image_path = self.df.loc[index, "image_path"]
        characteristic_vectors_str = self.df.loc[index, "characteristic_vectors"]

        # Konwersja wektorów charakterystycznych
        characteristic_vectors = ast.literal_eval(characteristic_vectors_str)  
        characteristic_vectors = np.array(characteristic_vectors, dtype=np.float32)

        # Ładowanie obrazu
        image = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Image not found: {image_path}")
        
        # Konwersja do RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Przekształcenia obrazu
        if self.transform:
            image = self.transform(Image.fromarray(image))

        # Skalowanie wektorów
        original_height, original_width = 64, 64  # Docelowe wymiary po transformacji
        characteristic_vectors[:, 0] *= (image.shape[2] / original_width)
        characteristic_vectors[:, 1] *= (image.shape[1] / original_height)

        return image, label, characteristic_vectors
        

In [8]:
# import ast

# transform = transforms.Compose([
#     transforms.Resize((64,64)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Values calculated from the ImageNet dataset
# ])


# def parse_vectors(vector_str):
#     # Remove all brackets and split by whitespace
#     cleaned_str = vector_str.replace('[', '').replace(']', '')
#     # Split into numbers and convert to float
#     numbers = [float(x) for x in cleaned_str.split() if x.strip()]
#     # Reshape into pairs (assuming each vector is 2D)
#     return np.array(numbers).reshape(-1, 2)


# class Dataset(Dataset):
#     def __init__(self, root_dir, csv_file, transform=None):
#         self.root_dir = root_dir
#         self.csv_file = csv_file
#         self.transform = transform
#         self.df = pd.read_csv(self.csv_file)
#         # self.classes = sorted(np.unique(self.df["class_index"]))
#         self.classes = sorted(np.unique(self.df["label"]))
#         self.class_names = sorted(np.unique(self.df["class_name"]))

#     def __len__(self):
#         dataset_len = len(self.df)
#         return dataset_len
        
#     def __getitem__(self, index):
#         if index >= len(self):
#             raise IndexError("Index out of bounds")
        
#         row = self.df.iloc[index]
#         # label = row["class_index"]
#         label = row["label"]
        
#         # Fix path construction
#         csv_path = row["image_path"]
#         # Remove any existing prefix paths to get just the relative part (letter/filename)
#         file_part = csv_path.split('/')[-2:]  # Gets ['F', 'F0011_test.jpg']
#         image_path = os.path.join(self.root_dir, *file_part)
        
#         characteristic_vectors = parse_vectors(row["characteristic_vectors"])
        
#         # Load and process image
#         image = cv2.imread(image_path, cv2.IMREAD_COLOR)
#         if image is None:
#             raise FileNotFoundError(f"Image not found: {image_path}")
        
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#         if self.transform:
#             image = self.transform(Image.fromarray(image))
            
#         # Scale vectors
#         original_height, original_width = 64, 64
#         characteristic_vectors[:, 0] *= (image.shape[2] / original_width)
#         characteristic_vectors[:, 1] *= (image.shape[1] / original_height)
        
#         return image, label, characteristic_vectors

In [9]:
# Train dataset and dataloader initialization
train_dataset = Dataset(root_dir="./newest_dataset/validated_train/",
                        csv_file= "./newest_dataset/validated_train_csv.csv",
                        transform=transform)

W0000 00:00:1738595385.441380   25386 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1738595385.463185   25410 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [10]:
# Sanity check
print("Dataset len:")
print(len(train_dataset))

print("\nDataset classes:")
print(train_dataset.classes)

print("\nRandom image from dataset:")
img, label, characteristic_vectors = train_dataset[random.randint(0, len(train_dataset)-1)]
image = img.permute(1, 2, 0).numpy()  # [C, H, W] -> [H, W, C]

print('\nImage size:')
print(img.shape)

# Denormalizing (there was normalization in transform)
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
image = image * std + mean
image = image.clip(0, 1)

plt.figure(figsize=(3,3))
plt.imshow(image)
plt.scatter(characteristic_vectors[:, 0], characteristic_vectors[:, 1], c='red')
plt.scatter([0, 0, img.shape[2], img.shape[2]], [0, img.shape[1], img.shape[1], 0], c="blue")  

plt.axis("off")
plt.title(label)
plt.show()

print("Characteristic vectors: ")
print(characteristic_vectors)


Dataset len:
26184

Dataset classes:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]

Random image from dataset:


SyntaxError: invalid syntax. Perhaps you forgot a comma? (<unknown>, line 1)

In [ ]:
# Train dataset and dataloader initialization
test_dataset = Dataset(root_dir="./newest_dataset/validated_test/",
                        csv_file= "./newest_dataset/validated_test_csv.csv",
                        transform=transform)

In [ ]:
print(pd.read_csv("./newest_dataset/validated_test_csv.csv").columns)

In [ ]:
# Sanity check
print("Dataset len:")
print(len(test_dataset))

print("\nDataset classes:")
print(test_dataset.classes)

print("\nRandom image from dataset:")
img, label, characteristic_vectors = test_dataset[random.randint(0, len(test_dataset)-1)]
image = img.permute(1, 2, 0).numpy()  # [C, H, W] -> [H, W, C]
print('\nImage size:')
print(img.shape)

# Denormalizing (there was normalization in transform)
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
image = image * std + mean
image = image.clip(0, 1)

plt.figure(figsize=(3,3))
plt.imshow(image)
plt.scatter(characteristic_vectors[:, 0], characteristic_vectors[:, 1], c='red')
plt.scatter([0, 0, img.shape[2], img.shape[2]], [0, img.shape[1], img.shape[1], 0], c="blue")  # Granice prostokąta

plt.axis("off")
plt.title(label)
plt.show()

print("Characteristic vectors: ")
print(characteristic_vectors)

## SPLITING DATA INTO TRAINING AND VALIDATION DATASETS

In [ ]:
train_len = round(0.8 * len(train_dataset))
val_len = len(train_dataset) - train_len
train_len, val_len, train_len + val_len == len(train_dataset)

In [ ]:
train_dataset_, val_dataset = random_split(train_dataset, [train_len, val_len])

In [ ]:
# Sanity check once more

print(f"Number of training samples: {len(train_dataset_)}")
print(f"Number of validation samples: {len(val_dataset)}")
print(f"Number of train classes: {len(train_dataset_.dataset.classes)}")
print(f"Number of val classes: {len(val_dataset.dataset.classes)}")
print(f"Number of test samples: {len(test_dataset)}") 


In [ ]:
print(f"Train dataset classes: {train_dataset.classes}")
print(f"Validation dataset classes: {train_dataset.classes}")

## DATALOADERS

In [ ]:
import os
num_workers = os.cpu_count()
print(f"Suggested num_workers: {num_workers}")

In [ ]:
import torch.multiprocessing as mp
mp.set_start_method('spawn', force=True)  # Use 'spawn' to safely initialize workers

train_dataloader = DataLoader(train_dataset_, 
                              batch_size=32, 
                              shuffle=True,
                              pin_memory=True)

val_dataloader = DataLoader(val_dataset,
                            batch_size=32,
                            shuffle=False,
                            pin_memory=True)

test_dataloader = DataLoader(test_dataset, 
                             batch_size=32, 
                             shuffle=False,
                             pin_memory=True)


In [ ]:
# Sanity check
def loader_sanity_check(loader):
    for batch_idx, (data, labels, characteristic_vectors) in enumerate(loader):
        print(f"Batch nr: {batch_idx}")
        print(f"Batch size: {data.shape[0]}")
        print(f"Data shape: {data.shape}")
        print(f"Image shape: {data[0].shape}")
        print(f"Characteristic vectors shape: {len(characteristic_vectors)}")
        print(f"Characteristic vectors: {characteristic_vectors}")
        print(f"Classes: {labels}")
        print(f"Num classes: {len(labels)}")
        break

In [ ]:
loader_sanity_check(train_dataloader)

In [ ]:
loader_sanity_check(val_dataloader)

In [ ]:
loader_sanity_check(test_dataloader)

## COMET_ML SETUP

In [ ]:
from comet_ml import Experiment
from utils2 import key

# Initialize Comet.ml experiment
experiment = Experiment(
    api_key=key,
    project_name="DLF-sign_letters_classification",
)

experiment.set_name("Multimodal model resnet")

## Define helper functions to logs gradients and weights

In [ ]:
def to_numpy(x):
    return x.detach().numpy()


def update_gradient_map(model, gradmap):
    for name, layer in zip(model._modules, model.children()):
        if "activ" in name:
            continue

        if not hasattr(layer, "weight"):
            continue

        wname = "%s/%s.%s" % ("gradient", name, "weight")
        bname = "%s/%s.%s" % ("gradient", name, "bias")

        gradmap.setdefault(wname, 0)
        gradmap.setdefault(bname, 0)

        gradmap[wname] += layer.weight.grad
        gradmap[bname] += layer.bias.grad

    return gradmap


def log_gradients(gradmap, step):
    for k, v in gradmap.items():
        experiment.log_histogram_3d(to_numpy(v), name=k, step=step)


def log_weights(model, step):
    for name, layer in zip(model._modules, model.children()):
        if "activ" in name:
            continue

        if not hasattr(layer, "weight"):
            continue

        wname = "%s.%s" % (name, "weight")
        bname = "%s.%s" % (name, "bias")

        experiment.log_histogram_3d(to_numpy(layer.weight), name=wname, step=step)
        experiment.log_histogram_3d(to_numpy(layer.bias), name=bname, step=step)

## MULTIMODAL MODEL

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class MultimodalModel(nn.Module):
    def __init__(self, num_classes, num_landmarks=21):
        super().__init__()
       
        # Image processing branch (ResNet50)
        self.base_model = models.resnet50(pretrained=True)
        self.base_model = nn.Sequential(*list(self.base_model.children())[:-1])
       
        # Freeze feature extractor
        for param in self.base_model.parameters():
            param.requires_grad = False
        
        # Image branch fully connected layers
        self.fc_image = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.LayerNorm(1024),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.LayerNorm(512),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3)
        )
       
        # Landmark processing branch
        self.fc_landmarks = nn.Sequential(
            nn.Linear(num_landmarks * 2, 128),
            nn.ReLU(),
            nn.LayerNorm(128),
            nn.Dropout(0.2),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3)
        )
       
        # Combined classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
       
    def forward(self, images, landmarks):
        # Process images through ResNet
        x = self.base_model(images)
        x = torch.flatten(x, 1)
        image_features = self.fc_image(x)
       
        # Process landmark features
        batch_size = landmarks.size(0)
        landmarks_flat = landmarks.view(batch_size, -1).float()
        landmark_features = self.fc_landmarks(landmarks_flat)
       
        # Combine features
        combined = torch.cat((image_features, landmark_features), dim=1)
       
        # Final classification
        output = self.classifier(combined)
        return output

In [ ]:
num_classes = len(train_dataset_.dataset.classes)
model = MultimodalModel(num_classes=num_classes)
model.to(device)

## MODEL PARAMETERS

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print(f"All parameters: {total_params}")
print(f"Frozen parameters: {frozen_params}")
print(f"Trainable parameters: {trainable_params}")


In [ ]:
learning_rate = 0.0001
batch_size = 32
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4) # https://paperswithcode.com/method/weight-decay
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3) # https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
num_epochs = 15
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

hyper_params = {"batch_size": batch_size, "num_epochs": num_epochs, "learning_rate": learning_rate}
# experiment.log_parameters(hyper_params)

In [ ]:
def train_model(device, model, train_loader, val_loader, criterion, optimizer, num_epochs, scheduler, resume_path=None, early_stopping_patience=5):
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')  
    # epochs_no_improve = 0

    # Load checkpoint if provided
    start_epoch = 0
    # if resume_path:
    #     checkpoint = torch.load(resume_path)
    #     model.load_state_dict(checkpoint['model_state_dict'])
    #     optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    #     start_epoch = checkpoint['epoch'] + 1
    #     print(f"Resuming training from epoch {start_epoch}")

    model.to(device)
    print(f"Starting Training")

    for epoch in range(start_epoch, num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        train_correct = 0
        train_total = 0

        gradmap = {}

        for batch_idx, (images, labels, characteristic_vectors) in enumerate(train_loader):
            # Convert characteristic vectors to tensor
            images, labels, characteristic_vectors = images.to(device), labels.to(device), characteristic_vectors.to(device) 

            optimizer.zero_grad()
            outputs = model(images, characteristic_vectors)  # Pass both images and characteristic vectors

            loss = criterion(outputs, labels)
            loss.backward()

            # Update gradient map
            gradmap = update_gradient_map(model, gradmap)

            optimizer.step()

            # Accumulate batch loss
            running_loss += loss.item()

            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

            # Log every 100 batches
            if batch_idx % 100 == 0:
                print(f'Epoch: {epoch + 1}, Batch: {batch_idx}, Loss: {loss.item():.4f}')
                experiment.log_metric("train_loss", loss.item(), step=epoch * len(train_loader) + batch_idx)

        # Log gradients
        log_gradients(gradmap, epoch)

        # Calculate epoch metrics
        epoch_loss = running_loss / len(train_loader.dataset)  # Average loss per sample
        train_accuracy = 100. * train_correct / train_total
        train_losses.append(epoch_loss)

        # Log epoch metrics
        experiment.log_metric("epoch_train_loss", epoch_loss, step=epoch)
        experiment.log_metric("epoch_train_accuracy", train_accuracy, step=epoch)

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        print("Validating...")
        with torch.inference_mode():
            for images, labels, characteristic_vectors in val_loader:
                images, labels, characteristic_vectors = images.to(device), labels.to(device), characteristic_vectors.to(device)
                outputs = model(images, characteristic_vectors)  
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_epoch_loss = val_loss / len(val_loader.dataset)  # Average loss per sample
        val_accuracy = 100. * val_correct / val_total
        val_losses.append(val_epoch_loss)

        print(f'Epoch [{epoch + 1}/{num_epochs}]')
        print(f'Training Loss: {epoch_loss:.4f}, Training Accuracy: {train_accuracy:.2f}%')
        print(f'Validation Loss: {val_epoch_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%')
        print('-' * 60)

        # Log validation metrics
        experiment.log_metric("val_loss", val_epoch_loss, step=epoch)
        experiment.log_metric("val_accuracy", val_accuracy, step=epoch)

        # Adjust learning rate using scheduler
        scheduler.step(val_epoch_loss)

        # Log weights
        log_weights(model, epoch)

        # Save best model checkpoint
        if val_epoch_loss < best_val_loss:
            best_val_loss = val_epoch_loss
            best_epoch = epoch
            epochs_no_improve = 0

            best_checkpoint_path = f"./models/multimodal_model/checkpoints/best_model.pt"
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict()
            }, best_checkpoint_path)

            print(f"Best model updated and saved at: {best_checkpoint_path}")
        else:
            epochs_no_improve += 1

        # Early stopping check
        if epochs_no_improve >= early_stopping_patience:
            print(f"Early stopping triggered. No improvement in validation loss for {early_stopping_patience} consecutive epochs.")
            break

        # Save model checkpoint
        checkpoint_path = f"./models/multimodal_model/checkpoints/Epoch {epoch+1}.pt"
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict()
        }, checkpoint_path)

        print(f"Model checkpoint saved at: {checkpoint_path}")

        model_path = f"./models/multimodal_model/checkpoints/model epoch {epoch+1}"
        weights_path = f"./models/multimodal_model/checkpoints/model weights epoch {epoch+1}"

        torch.save(model, model_path)
        torch.save(model.state_dict(), weights_path)

        print(f"Model checkpoint saved at: {checkpoint_path}")

    return train_losses, val_losses


In [ ]:
# Test loop
def test_model(device, model, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
    
        for i, (images, labels, characteristic_vectors) in enumerate(test_loader):
            images = images.float().to(device)
            labels = labels.to(device)
            characteristic_vectors = characteristic_vectors.float().to(device)

            # images, labels, characteristic_vectors = images.to(device), labels.to(device), characteristic_vectors.to(device)
            outputs = model(images, characteristic_vectors)  # Pass both images and characteristic vectors
            loss = criterion(outputs, labels)

            test_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_loss /= len(test_loader.dataset)
    test_accuracy = 100 * correct / total

    print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%')

    # # Log test metrics
    # experiment.log_metric("test_loss", test_loss)
    # experiment.log_metric("test_accuracy", test_accuracy)

    return test_loss, test_accuracy


## TRAINING

In [ ]:
# train_losses, val_losses = train_model(device, model, train_dataloader, val_dataloader, criterion, optimizer, num_epochs, scheduler)

In [ ]:
# # Saving model and weights
# torch.save(model, "./models/multimodal_model2")
# torch.save(model.state_dict(), "./models/multimodal_model_weights2")

## Log the Model to Comet

In [ ]:
from comet_ml.integration.pytorch import log_model

log_model(experiment, model, "multimodal model")

## End Experiment

In [ ]:
experiment.end()

## TESTING

In [ ]:
test_loss, test_accuracy = test_model(device, model, test_dataloader, criterion)

# TESTING ON A DIFFERENT DATASET

In [ ]:
!pwd

In [ ]:
!ls models

In [ ]:
# model = torch.load(f="./models/multimodal_model/multimodal_model", weights_only=False)
# model.to(device)


num_classes = len(train_dataset_.dataset.classes)

model = MultimodalModel(num_classes)
checkpoint = torch.load("./models/multimodal/best_model.pt", weights_only=True)

model.load_state_dict(checkpoint['model_state_dict'])

model.to(device)

In [ ]:
import ast

transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Values calculated from the ImageNet dataset
])


def parse_vectors(vector_str):
    cleaned_str = vector_str.replace('[', '').replace(']', '')
    numbers = [float(x) for x in cleaned_str.split() if x.strip()]
    return np.array(numbers).reshape(-1, 2)


class Dataset(Dataset):
    def __init__(self, root_dir, csv_file, transform=None):
        self.root_dir = root_dir
        self.csv_file = csv_file
        self.transform = transform
        self.df = pd.read_csv(self.csv_file)
        self.classes = sorted(np.unique(self.df["class_index"]))
        self.class_names = sorted(np.unique(self.df["class_name"]))
        
    def __getitem__(self, index):
        if index >= len(self):
            raise IndexError("Index out of bounds")
        
        row = self.df.iloc[index]
        label = row["class_index"]
        
        # Fix path construction
        csv_path = row["image_path"]
        # Remove any existing prefix paths to get just the relative part (letter/filename)
        file_part = csv_path.split('/')[-2:]  # Gets ['F', 'F0011_test.jpg']
        image_path = os.path.join(self.root_dir, *file_part)
        
        characteristic_vectors = parse_vectors(row["characteristic_vectors"])
        
        # Load and process image
        image = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Image not found: {image_path}")
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(Image.fromarray(image))
            
        # Scale vectors
        original_height, original_width = 64, 64
        characteristic_vectors[:, 0] *= (image.shape[2] / original_width)
        characteristic_vectors[:, 1] *= (image.shape[1] / original_height)
        
        return image, label, characteristic_vectors

In [ ]:
# Train dataset and dataloader initialization
validated_test_dataset = Dataset(root_dir="./newest_dataset/validated_test/",
                        csv_file= "./newest_dataset/validated_test_csv.csv",
                        transform=transform)

In [ ]:
# # Train dataset and dataloader initialization
# validated_test_dataset = Dataset(root_dir="./dataset/test/",
#                         csv_file= "./newest_dataset/validated_test_csv.csv",
#                         transform=transform)

In [ ]:
# Sanity check
print("Dataset len:")
print(len(validated_test_dataset))

print("\nDataset classes:")
print(validated_test_dataset.classes)

print("\nRandom image from dataset:")
img, label, characteristic_vectors = validated_test_dataset[random.randint(0, len(validated_test_dataset)-1)]
image = img.permute(1, 2, 0).numpy()  # [C, H, W] -> [H, W, C]

# Denormalizing (there was normalization in transform)
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
image = image * std + mean
image = image.clip(0, 1)

plt.figure(figsize=(3,3))
plt.imshow(image)
plt.axis("off")
plt.title(label)
plt.show()

print("Characteristic vectors: ")
print(characteristic_vectors)


In [ ]:
validated_test_dataset_loader = DataLoader(validated_test_dataset, 
                             batch_size=32, 
                             shuffle=False, 
                             num_workers=0)


In [ ]:
criterion = nn.CrossEntropyLoss()
test_loss, test_accuracy = test_model(device, model, validated_test_dataset_loader, criterion)

In [ ]:
# Test loop
def test_model(device, model, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
    
        for i, (images, labels, characteristic_vectors) in enumerate(test_loader):
            images = images.float().to(device)
            labels = labels.to(device)
            characteristic_vectors = characteristic_vectors.float().to(device)

            # images, labels, characteristic_vectors = images.to(device), labels.to(device), characteristic_vectors.to(device)
            outputs = model(images, characteristic_vectors)  # Pass both images and characteristic vectors
            loss = criterion(outputs, labels)

            print("Number of unique labels in dataset:", len(set(labels)))  # or however you access your labels
            print("Current num_classes in model:", num_classes)

            test_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_loss /= len(test_loader.dataset)
    test_accuracy = 100 * correct / total

    print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%')

    # # Log test metrics
    # experiment.log_metric("test_loss", test_loss)
    # experiment.log_metric("test_accuracy", test_accuracy)

    return test_loss, test_accuracy

test_model(device, model, test_dataloader, criterion)

In [ ]:
def test_model(device, model, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    
    num_classes = len(test_loader.dataset.classes)
    confusion_matrix = torch.zeros(num_classes, num_classes)
    
    with torch.inference_mode():
        for i, (images, labels, characteristic_vectors) in enumerate(test_loader):
            # inputs, labels = inputs.to(device), labels.to(device)

            images = images.float().to(device)
            labels = labels.to(device)
            characteristic_vectors = characteristic_vectors.float().to(device)
            
            outputs = model(images, characteristic_vectors)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            print("Number of unique labels in dataset:", len(set(labels)))  # or however you access your labels
            print("Current num_classes in model:", num_classes)
            
            for t, p in zip(labels.view(-1), predicted.view(-1)):
                confusion_matrix[t.long(), p.long()] += 1

    test_loss /= len(test_loader)
    test_accuracy = 100 * correct / total
    
    print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%')
    
    print("\nPer-class accuracy:")
    class_names = test_loader.dataset.classes
    for i in range(num_classes):
        class_accuracy = 100 * confusion_matrix[i, i] / confusion_matrix[i].sum()
        print(f'Class {class_names[i]}: {class_accuracy:.2f}%')
        experiment.log_metric(f"class_{class_names[i]}_accuracy", class_accuracy)
    
    return test_loss, test_accuracy, confusion_matrix


test_loss, test_accuracy = test_model(device, model, test_dataloader, criterion)

In [ ]:

def plot_confusion_matrix(confusion_matrix, class_names):
    cm_percent = confusion_matrix.numpy()
    cm_percent = cm_percent / cm_percent.sum(axis=1, keepdims=True) * 100

    plt.figure(figsize=(15, 15))

    sns.heatmap(cm_percent, 
                annot=True, 
                fmt='.1f', 
                xticklabels=class_names,
                yticklabels=class_names,
                cmap='Blues',
                vmin=0,
                vmax=100)
    
    plt.title('Confusion Matrix (%)')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    
    plt.savefig('confusion_matrix.png')
    plt.show()

    print("\nMost confused pairs (True -> Predicted):")
    n_classes = len(class_names)
    confused_pairs = []
    
    for i in range(n_classes):
        for j in range(n_classes):
            if i != j and cm_percent[i, j] > 1:  # More than 1% confusion
                confused_pairs.append((
                    class_names[i], 
                    class_names[j], 
                    cm_percent[i, j]
                ))
    
    confused_pairs.sort(key=lambda x: x[2], reverse=True)
    
    for true_class, pred_class, percent in confused_pairs[:10]:
        print(f"{true_class} -> {pred_class}: {percent:.1f}%")

class_names = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
plot_confusion_matrix(confusion_matrix, class_names)